# E03. Evaluate Lineups
- This evaluates contest results
- Type: Evaluation
- Run Frequency: Irregular
- Sources:
    - DraftKings
- Created: 3/30/2024
- Updated: 2/23/2026

### Imports

In [1]:
from DataImports import *

### Function

This assigns payouts and projected payouts to contest entries and my portfolio of possible entries

In [2]:
def evaluate_portfolio_lineups(contestKey):
    ### Portfolio Lineups
    def prepare_my_lineups(contestKey):
        # Read
        portfolio_lineup_df = pd.read_csv(os.path.join(baseball_path, "C02. Optimization", "4. Portfolio Lineups", f"Portfolio Lineups {contestKey}.csv"))

        # Remove parentheses and numbers from player names in portfolio lineups to merge with player scores
        for col in portfolio_lineup_df.select_dtypes(include='object').columns:
            portfolio_lineup_df[col] = portfolio_lineup_df[col].str.replace(r'\(\d+\)', '', regex=True)

        ### Player Results
        # Read
        player_score_df = pd.read_csv(os.path.join(baseball_path, "A09. DraftKings", "6. Player Results", f"Player Results {contestKey}.csv"), usecols=['Player', 'FPTS', '%Drafted'], encoding='iso-8859-1')

        # Note: player_score_df contains name only in "Player" column, not "Name + ID," so we have to merge on name only. This can lead to some issues with duplicate names (Will Smith, for instance).
        # I'm defauling to picking the most likely player (using %Drafted) if there are duplicate names. This could perhaps be improved by pulling in position from Player Results and merging on name + position.
        # This wouldn't matter much, however. Even choosing the least likely or lowest scoring duplicate player barely matters.
        player_score_df = player_score_df.sort_values('%Drafted', ascending=False).drop_duplicates(subset=['Player'], keep='first')
        player_score_df.drop(columns=['%Drafted'], inplace=True)

        # For each position, merge the player's point total
        points_list = []
        for pos in ['P', 'P.1', 'C', '1B', '2B', '3B', 'SS', 'OF', 'OF.1', 'OF.2']:
            # Merges on name only, remove numeric characters
            portfolio_lineup_df[pos] = portfolio_lineup_df[pos].str.replace(r'[(0-9)]', "", regex=True)
            portfolio_lineup_df[pos] = portfolio_lineup_df[pos].str.rstrip()

            # Merge with points
            portfolio_lineup_df = portfolio_lineup_df.merge(player_score_df, left_on=pos, right_on='Player', how='left')
            points_name = 'FPTS' + "_" + pos
            points_list.append(points_name)
            portfolio_lineup_df.drop(columns={'Player'},inplace=True)
            portfolio_lineup_df.rename(columns={'FPTS':points_name},inplace=True)

        # Sum points across all positions for each lineup
        portfolio_lineup_df['Points'] = portfolio_lineup_df[points_list].sum(axis=1)
        # Add entry information
        portfolio_lineup_df['EntryId'] = "Me"
        portfolio_lineup_df['EntryName'] = "Me"
        portfolio_lineup_df['TimeRemaining'] = 0
        portfolio_lineup_df['Lineup'] = ""


        return portfolio_lineup_df[['EntryId', 'EntryName', 'TimeRemaining', 'Points', 'Lineup', 'P', 'P.1', 'C', '1B', '2B', '3B', 'SS', 'OF', 'OF.1', 'OF.2', 'stack']]

    # Run
    portfolio_lineup_df = prepare_my_lineups(contestKey)


    ### Entry Results
    # Read
    entry_result_df = pd.read_csv(os.path.join(baseball_path, "A09. DraftKings", "5. Entry Results", f"Entry Results {contestKey}.csv"), encoding='iso-8859-1')

    # Merge portfolio lineups with entry results 
    all_lineup_df = pd.concat([portfolio_lineup_df, entry_result_df], ignore_index=True)

    # Sort by Points descending and assign ranks
    all_lineup_df['Projected Rank'] = all_lineup_df['Points'].rank(method='min', ascending=False).astype(int)

    # Sort the DataFrame by rank
    all_lineup_df = all_lineup_df.sort_values('Projected Rank').reset_index(drop=True)


    ### Payouts
    # Read
    payout_df = pd.read_csv(os.path.join(baseball_path, "A09. DraftKings", "3. Payouts", f"Payouts {contestKey}.csv"), encoding='iso-8859-1')

    # First, create a mapping of each position to its payout
    # For ranges, expand them into individual positions
    payout_mapping = {}
    for _, row in payout_df.iterrows():
        for pos in range(row['minPosition'], row['maxPosition'] + 1):
            payout_mapping[pos] = row['payoutDescription']

    # Calculate Actual Payouts
    # Now, assign a preliminary payout based on rank
    all_lineup_df['Payout'] = all_lineup_df['Rank'].map(payout_mapping)

    # Convert payoutDescription to numeric if needed
    all_lineup_df['Payout'] = all_lineup_df['Payout'].replace('[\$,]', '', regex=True).astype(float)

    # Group by rank to handle ties
    all_lineup_df['Payout'] = all_lineup_df.groupby('Rank')['Payout'].transform('mean')

    # Fill missing payouts with 0s
    all_lineup_df['Payout'] = all_lineup_df['Payout'].fillna(0)


    # Calculate Projected Payouts
    # Now, assign a preliminary payout based on rank
    all_lineup_df['Projected Payout'] = all_lineup_df['Projected Rank'].map(payout_mapping)

    # Handle ties: average the payouts for tied ranks
    # Convert payoutDescription to numeric if needed
    all_lineup_df['Projected Payout'] = all_lineup_df['Projected Payout'].replace('[\$,]', '', regex=True).astype(float)

    # Group by rank to handle ties
    all_lineup_df['Projected Payout'] = all_lineup_df.groupby('Projected Rank')['Projected Payout'].transform('mean')

    # Fill missing payouts with 0s
    all_lineup_df['Projected Payout'] = all_lineup_df['Projected Payout'].fillna(0)


    ### Contest Guides
    # Read
    contest_guide_df = pd.read_csv(os.path.join(baseball_path, "B03. Contest Guides", f"Contest Guide {contestKey}.csv"), encoding='iso-8859-1')        
    
    # Extract entry fee
    all_lineup_df['entryFee'] = float(contest_guide_df.loc[0, 'entryFee'])

    # Calculate ROI
    all_lineup_df['ROI'] = (all_lineup_df['Payout'] - all_lineup_df['entryFee']) / all_lineup_df['entryFee']
    all_lineup_df['Projected ROI'] = (all_lineup_df['Projected Payout'] - all_lineup_df['entryFee']) / all_lineup_df['entryFee']

    # Calculate percentile
    all_lineup_df['Projected Percentile'] = (1 - (all_lineup_df['Projected Rank'] - 1) / len(all_lineup_df))
    all_lineup_df['Projected 95th Percentile'] = (all_lineup_df['Projected Percentile'] >= 0.95).astype(int)
    all_lineup_df['Projected 99th Percentile'] = (all_lineup_df['Projected Percentile'] >= 0.99).astype(int)

    # Add contestKey
    all_lineup_df['contestKey'] = contestKey


    return all_lineup_df

### Sample

In [3]:
# Create list of contestKeys with available guides
guide_contestKey_list = os.listdir(os.path.join(baseball_path, "B03. Contest Guides"))
guide_contestKey_list = [int(key.replace(".csv", "").split(" ")[2]) for key in guide_contestKey_list]

# Create list of contestKeys with available results
result_contestKey_list = os.listdir(os.path.join(baseball_path, "A09. DraftKings", "6. Player Results"))
result_contestKey_list = [int(key.replace(".csv", "").split(" ")[2]) for key in result_contestKey_list]

# Create list of contestKeys with available Payouts
payout_contestKey_list = os.listdir(os.path.join(baseball_path, "A09. DraftKings", "3. Payouts"))
payout_contestKey_list = [int(key.replace(".csv", "").split(" ")[1]) for key in payout_contestKey_list]

# Create a list of contestKeys with optimized lineups
lineup_contestKey_list = os.listdir(os.path.join(baseball_path, "C02. Optimization", "5. Lineups Ranked"))
lineup_contestKey_list = [int(key.replace(".csv", "").split(" ")[2]) for key in lineup_contestKey_list]

# Find the overlap 
available_contestKey_list = list(set(result_contestKey_list) & set(guide_contestKey_list) & set(payout_contestKey_list) & set(lineup_contestKey_list))

### Run

In [4]:
%%time
contest_results_list = Parallel(n_jobs=-1, verbose=1)(delayed(evaluate_portfolio_lineups)(contestKey) for contestKey in available_contestKey_list)
all_contest_results_df = pd.concat(contest_results_list, axis=0)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done 430 out of 430 | elapsed:    2.0s finished


CPU times: total: 1.89 s
Wall time: 2.54 s


### Evaluate

##### My Lineups

In [5]:
all_contest_results_df.query('EntryName == "Me"')[['Projected Payout', 'entryFee', 'Projected ROI', 'Projected Percentile', 'Projected 95th Percentile', 'Projected 99th Percentile']].describe()

,Projected Payout,entryFee,Projected ROI,Projected Percentile,Projected 95th Percentile,Projected 99th Percentile
count,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000,8600.000000
mean,5.375465,4.000000,0.343866,0.635337,0.097674,0.018372
std,18.852722,0.000000,4.713180,0.262691,0.296891,0.134301
min,0.000000,4.000000,-1.000000,0.000668,0.000000,0.000000
25%,0.000000,4.000000,-1.000000,0.445517,0.000000,0.000000
50%,0.000000,4.000000,-1.000000,0.686005,0.000000,0.000000
75%,8.000000,4.000000,1.000000,0.860944,0.000000,0.000000
max,600.000000,4.000000,149.000000,1.000000,1.000000,1.000000


##### Stack Performance

In [6]:
(all_contest_results_df
        .groupby('stack')['Projected ROI']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'Projected ROI', 'count': 'Num Entries'})
        .sort_values('Projected ROI', ascending=False)
        .reset_index())

,stack,Projected ROI,Num Entries
0,"[5, 1, 1, 1]",0.482691,1661
1,"[4, 2, 1, 1]",0.476204,1765
2,"[5, 3]",0.384089,1279
3,"[5, 2, 1]",0.265959,2804
4,"[4, 3, 1]",0.071494,1091


##### Top Entrants

In [7]:
all_contest_results_df['Entrant'] = all_contest_results_df['EntryName'].str.replace(r'\(.*?\)', '', regex=True).str.strip()
min_entries = 1000  # minimum number of entries required

# Aggregate mean projected payout, mean payout, and count
summary_df = (
    all_contest_results_df
    .groupby('Entrant')
    .agg(
        **{
            'Mean Projected Payout': ('Projected Payout', 'mean'),
            'Mean Payout': ('Payout', 'mean'),
            'Mean Projected ROI': ('Projected ROI', 'mean'),
            'Mean Projected Percentile': ('Projected Percentile', 'mean'),
            'Mean Projected 95th Percentile': ('Projected 95th Percentile', 'mean'),
            'Mean Projected 99th Percentile': ('Projected 99th Percentile', 'mean'),
            'Mean Points': ('Points', 'mean'),
            'Num Entries': ('Projected Payout', 'count') 
        }
    )
    .query('`Num Entries` >= @min_entries')  # filter by min_entries
    .sort_values('Mean Projected Payout', ascending=False)  # sort by projected payout
    .reset_index()
)

summary_df.head(25)

,Entrant,Mean Projected Payout,Mean Payout,Mean Projected ROI,Mean Projected Percentile,Mean Projected 95th Percentile,Mean Projected 99th Percentile,Mean Points,Num Entries
0,ImaDonk11,8.821953,8.905791,1.205488,0.501309,0.050130,0.019879,95.514520,1157
1,BCap888,8.014680,8.025043,1.003670,0.497439,0.056131,0.016408,96.334931,1158
2,edgelesscart,7.913163,7.939352,0.978291,0.536974,0.066161,0.020675,99.552963,1451
3,inittobinkit,7.831254,7.837790,0.957813,0.496508,0.057635,0.016637,94.791800,1683
4,drosenstock,7.773251,7.787855,0.943313,0.499312,0.056111,0.007686,95.611568,1301
5,Squizzward,6.842254,6.865728,0.710563,0.487005,0.060094,0.018779,94.646150,1065
6,KeefVape,6.286329,6.292788,0.571582,0.505871,0.057051,0.015070,98.546986,1858
7,Ddohnalek,5.854167,5.865036,0.463542,0.517209,0.067029,0.016304,98.145652,1104
8,sboynton10,5.701098,5.730040,0.425274,0.535732,0.060878,0.014471,98.957260,2004
9,lp305,5.562698,5.582540,0.390675,0.499460,0.062698,0.017460,94.876627,1260


Notes: 
- 95th percentile outperformed 99th percentile in first test
- More field lineups (30000 attempts) outperformed small group (10000 attempts)

Consider Adding:
- Optimized lineup support. Take lineups ranked version, top n by metric

Performance Tests:
- Field Lineups, unsorted, did poorly. Supports idea that I'm not just unilaterally overrating my lineups here, although this isn't definitive.
- Field Lineups, sorted, did great. Supports idea that my projections would have done very well in 2024 for determining lineup choice even if they weren't used in creation.
- Optimized Lineups, unsorted, did fantastic. Better than portfolio lineups, in fact. Not particularly bullish for the portfolio approach, but nice for the projections.

Conclusions:
- Code is likely correctly assigning ranks and payouts
- 2024 projections/lineups are very good. May be benefitting from too much insider information.
- Portfolio isn't definitely better than other approaches. Requires more testing.